In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Lokalny tryb testowania Qiskit Runtime

<span id="test-locally"></span>


<details>
<summary><b>Wersje pakietów</b></summary>

Kod na tej stronie został opracowany z użyciem poniższych wymagań.
Zalecamy korzystanie z tych lub nowszych wersji.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-aer~=0.17
```
</details>
Użyj lokalnego trybu testowania (dostępnego w `qiskit-ibm-runtime` w wersji 0.22.0 lub nowszej), aby testować programy przed ich dopracowaniem i wysłaniem na prawdziwy sprzęt kwantowy. Po zweryfikowaniu programu w lokalnym trybie testowania wystarczy zmienić nazwę Backend, aby uruchomić go na QPU.

Aby skorzystać z lokalnego trybu testowania, wskaż jeden z fałszywych backendów z ``qiskit_ibm_runtime.fake_provider`` lub podaj Backend Qiskit Aer podczas tworzenia instancji prymitywu Qiskit Runtime albo Session.

- **Fałszywe backendy**: [Fałszywe backendy](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider) z ``qiskit_ibm_runtime.fake_provider`` naśladują zachowanie QPU IBM&reg; poprzez używanie migawek QPU. Migawki QPU zawierają ważne informacje o QPU, takie jak mapa sprzężeń, bazowe Gate'y i właściwości qubitów, co jest przydatne do testowania Transpilera i przeprowadzania symulacji QPU z szumem. Model szumu z migawki jest automatycznie stosowany podczas symulacji.
- **Symulator Aer**: Symulatory z [Qiskit Aer](/guides/simulate-with-qiskit-aer) zapewniają wydajniejszą symulację, która obsługuje większe Circuit i [niestandardowe modele szumu](/guides/build-noise-models). Podczas używania `AerSimulator` w lokalnym trybie testowania dostępna jest lista opcji metod symulacji. Zobacz [przykład trybu symulacji Clifford](#clifford-sim), który pokazuje, jak efektywnie symulować Circuit Clifford z dużą liczbą Qubitów.

    <details>
    <summary>**Lista metod symulacji dostępnych w Qiskit Aer**</summary>

    Więcej informacji znajdziesz w dokumentacji [`AerSimulator`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator).

    * ``"automatic"``: Domyślna metoda symulacji. Automatycznie wybiera metodę symulacji na podstawie Circuit i modelu szumu.

    * ``"statevector"``: Gęsta symulacja wektora stanu, która może próbkować wyniki pomiarów z *idealnych* Circuit z wszystkimi pomiarami na końcu. W przypadku symulacji z szumem każde ujęcie próbkuje losowo wybrany Circuit z szumem z modelu szumu.

    * ``"density_matrix"``: Symulacja macierzy gęstości, która może próbkować wyniki pomiarów z *zaszumionych* Circuit z wszystkimi pomiarami na końcu.

    * ``"stabilizer"``: Wydajny symulator stanów stabilizatora Clifford, który potrafi symulować zaszumione Circuit Clifford, jeśli wszystkie błędy w modelu szumu są również błędami Clifford.

    * ``"extended_stabilizer"``: Przybliżony symulator dla Circuit Clifford + T, oparty na rozkładzie stanu na rangowany stan stabilizatora. Liczba składników rośnie wraz z liczbą bramek nie-Clifford (T).

    * ``"matrix_product_state"``: Symulator wektora stanu oparty na sieciach tensorowych, który używa reprezentacji Matrix Product State (MPS) dla stanu. Można go stosować z obcinaniem lub bez obcinania wymiarów wiązań MPS, w zależności od opcji symulatora. Domyślnie brak obcinania.

    * ``"unitary"``: Gęsta symulacja macierzy unitarnej idealnego Circuit. Symuluje macierz unitarną samego Circuit, a nie ewolucję początkowego stanu kwantowego. Ta metoda obsługuje tylko Gate'y; nie obsługuje pomiarów, resetowania ani szumu.

    * ``"superop"``: Gęsta symulacja macierzy superoperatora dla idealnego lub zaszumionego Circuit. Symuluje macierz superoperatora samego Circuit, a nie ewolucję początkowego stanu kwantowego. Ta metoda potrafi symulować idealne i zaszumione Gate'y oraz resety, ale nie obsługuje pomiarów.

    * ``"tensor_network"``: Symulacja oparta na sieciach tensorowych obsługująca zarówno wektor stanu, jak i macierz gęstości. Obecnie dostępna wyłącznie dla GPU i akcelerowana przy użyciu interfejsów API cuQuantum `cuTensorNet`.
    </details>

> **Note:** - W lokalnym trybie testowania możesz podać wszystkie opcje Qiskit Runtime. Jednak wszystkie opcje poza `shots` są ignorowane podczas uruchamiania na lokalnym symulatorze.
> - Zaleca się zainstalowanie Qiskit Aer przed użyciem fałszywych backendów lub symulatorów Aer, uruchamiając `pip install qiskit-aer`. Fałszywe backendy używają symulatorów Aer pod spodem, jeśli są dostępne, aby skorzystać z ich wydajności.


## Przykład fałszywych backendów

In [1]:
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime.fake_provider import FakeManilaV2

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using FakeManilaV2
fake_manila = FakeManilaV2()
pm = generate_preset_pass_manager(backend=fake_manila, optimization_level=1)
isa_qc = pm.run(qc)

# You can use a fixed seed to get fixed results.
options = {"simulator": {"seed_simulator": 42}}
sampler = Sampler(mode=fake_manila, options=options)

result = sampler.run([isa_qc]).result()

## Przykłady AerSimulator {#aerSimulator-examples}

Przykład z Session, bez szumu:

In [2]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import Session, SamplerV2 as Sampler

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using AerSimulator.
# Session syntax is supported but ignored because local mode doesn't support sessions.
aer_sim = AerSimulator()
pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
with Session(backend=aer_sim) as session:
    sampler = Sampler(mode=session)
    result = sampler.run([isa_qc]).result()

Aby symulować z szumem, wskaż QPU (sprzęt kwantowy) i prześlij go do Aer. Aer buduje model szumu na podstawie danych kalibracyjnych z tego QPU i tworzy instancję Backend Aer z tym modelem. Jeśli wolisz, możesz [zbudować model szumu](/guides/build-noise-models).

> **Caution:** Na QPU może mieć wpływ wiele rodzajów szumu. Model szumu Qiskit Aer używany tutaj symuluje tylko niektóre z nich i w związku z tym jest prawdopodobnie mniej dotkliwy niż szum na prawdziwym QPU.
> 
> Szczegółowe informacje o tym, jakie błędy są uwzględniane podczas inicjalizacji modelu szumu z QPU, znajdziesz w dokumentacji API Aer [`NoiseModel`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.noise.NoiseModel.html#qiskit_aer.noise.NoiseModel.from_backend).

Przykład z szumem:

In [3]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler, QiskitRuntimeService

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

service = QiskitRuntimeService()

# Specify a QPU to use for the noise model
real_backend = service.backend("ibm_fez")
aer = AerSimulator.from_backend(real_backend)

# Run the sampler job locally using AerSimulator.
pm = generate_preset_pass_manager(backend=aer, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer)
result = sampler.run([isa_qc]).result()

<span id="clifford-sim"></span>
### Symulacja Clifford
Ponieważ Circuit Clifford można efektywnie symulować z weryfikowalnymi wynikami, symulacja Clifford jest bardzo przydatnym narzędziem. Szczegółowy przykład znajdziesz w artykule [Efektywna symulacja Circuit stabilizatora za pomocą prymitywów Qiskit Aer](/guides/simulate-stabilizer-circuits).

Przykład:

In [4]:
import numpy as np
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import SamplerV2 as Sampler

n_qubits = 500  # <---- note this uses 500 qubits!
circuit = efficient_su2(n_qubits)
circuit.measure_all()

rng = np.random.default_rng(1234)
params = rng.choice(
    [0, np.pi / 2, np.pi, 3 * np.pi / 2],
    size=circuit.num_parameters,
)

# Tell Aer to use the stabilizer (Clifford) simulation method
aer_sim = AerSimulator(method="stabilizer")

pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer_sim)
result = sampler.run([isa_qc]).result()

## Następne kroki
> **Tip:** - Przejrzyj szczegółowe [przykłady prymitywów.](/guides/primitives-examples)
>     - Przeczytaj [Migracja do prymitywów V2](/guides/v2-primitives).
>     - Ćwicz korzystanie z prymitywów, pracując nad [lekcją funkcji kosztu](/learning/courses/variational-algorithm-design/cost-functions) w IBM Quantum Learning.
>     - Dowiedz się, jak transpilować lokalnie, w sekcji [Transpiler](/guides/transpile).
>     - Wypróbuj samouczek [Porównaj ustawienia Transpilera](/guides/circuit-transpilation-settings#compare-transpiler-settings).